# Estudo Prático de Pipelines do Hugging Face

A biblioteca `transformers` da Hugging Face oferece APIs em dois níveis de abstração.

A API de Alto Nível para uso de modelos open-source em tarefas típicas de inferência é chamada de **pipelines**. É incrivelmente simples e direta de usar.

Sintaxe básica para criar um pipeline:

`meu_pipeline = pipeline("tarefa_desejada")`

Execução da inferência:

`resultado = meu_pipeline(minha_entrada)`

No final deste notebook, há uma lista de referência com os principais pipelines disponíveis.

## Conceitos Fundamentais: Treinamento vs. Inferência

Conceitos essenciais ao trabalhar com modelos de Ciência de Dados e IA:

### 1. Treinamento (Training) e Fine-Tuning
- **Treinamento:** Etapa em que o modelo aprende com os dados e atualiza suas configurações internas (parâmetros/pesos).
- **Ajuste Fino (Fine-Tuning):** Processo de adaptar um modelo pré-treinado para uma tarefa específica com novos dados.

### 2. Inferência (Inference)
- **Inferência:** Utilização de um modelo *já treinado* para gerar previsões ou saídas a partir de novas entradas (também chamada de execução).
- O uso de APIs para LLMs (como GPT, Claude e Gemini) é um exemplo de inferência. O "P" em GPT significa *Pre-trained* (Pré-treinado).

> **Nota:** A API de `pipelines` do Hugging Face é focada exclusivamente em **inferência** (execução de modelos já treinados).

In [1]:
# As instalações do pip devem ficar na primeira linha.
# Se o seu Kernel reiniciar, você precisará executar isto novamente.

!pip install -q --upgrade datasets==3.6.0 transformers==4.57.6

In [2]:
# Vamos verificar a GPU - deve ser uma Tesla T4

gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Não conectado a uma GPU')
else:
  print(gpu_info)
  if gpu_info.find('Tesla T4') >= 0:
    print("Sucesso - Conectado a uma T4")
  else:
    print("NÃO CONECTADO A UMA T4")

Sun Jul 26 21:56:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# Importações

import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import pipeline
from diffusers import DiffusionPipeline
from datasets import load_dataset
import soundfile as sf
from IPython.display import Audio

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


## Configuração da Conta e Token do Hugging Face

Para utilizar certos modelos ou salvar artefatos na plataforma:

1. Acessar a conta em [Hugging Face](https://huggingface.co).
2. Criar um token de acesso de API no menu `Settings -> Access Tokens` (garantindo permissões de leitura/escrita conforme necessário).
3. Adicionar o token (`HF_TOKEN`) na aba de Segredos (*Secrets* / ícone de chave na barra lateral esquerda do Colab) e conceder acesso ao notebook.

In [4]:
hf_token = userdata.get('HF_TOKEN')
if hf_token and hf_token.startswith("hf_"):
  print("A chave do HF parece correta até agora")
else:
  print("A chave do HF não está configurada - por favor clique no ícone de chave na barra lateral esquerda")
login(hf_token, add_to_git_credential=True)

A chave do HF parece correta até agora


## Como Funcionam os Pipelines

Os pipelines facilitam a execução de inferência para tarefas comuns sem necessidade de configurar tokenizadores ou modelos manualmente.

### Etapas de Uso:

**Passo 1:** Criar a instância do pipeline
```python
meu_pipeline = pipeline(task, model=xx, device=xx)
```
- Se o parâmetro `model` não for especificado, o Hugging Face utilizará o modelo padrão configurado para a tarefa.
- Defina `device="cuda"` para utilizar GPU NVIDIA (ex: T4 no Colab) ou `device="mps"` no macOS com Apple Silicon.

**Passo 2:** Executar a inferência chamando a instância
```python
meu_pipeline(entrada1)
meu_pipeline(entrada2)
```

In [5]:
# Análise de Sentimento

my_simple_sentiment_analyzer = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment", device="cuda")
result = my_simple_sentiment_analyzer("Estou super animado para dominar o estudo de LLMs!")
print(result)

Device set to use cuda


[{'label': '5 stars', 'score': 0.7920127511024475}]


In [6]:
result = my_simple_sentiment_analyzer("Eu deveria estar mais animado com este aprendizado de LLMs!")
print(result)

[{'label': '3 stars', 'score': 0.3635872006416321}]


In [7]:
better_sentiment = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment", device="cuda")
result = better_sentiment("Este recurso de pipelines é excelente e facilita muito o trabalho!")
print(result)

Device set to use cuda


[{'label': '5 stars', 'score': 0.7731018662452698}]


In [8]:
# Reconhecimento de Entidades Nomeadas (NER)

ner = pipeline("ner", model="Babelscape/wikineural-multilingual-ner", device="cuda")
result = ner("Engenheiros de IA estão estudando a biblioteca Hugging Face no Google Colab em São Paulo.")
for entity in result:
  print(entity)

Device set to use cuda


{'entity': 'B-MISC', 'score': np.float32(0.87573886), 'index': 12, 'word': 'Hu', 'start': 47, 'end': 49}
{'entity': 'I-MISC', 'score': np.float32(0.9771697), 'index': 13, 'word': '##gging', 'start': 49, 'end': 54}
{'entity': 'I-MISC', 'score': np.float32(0.97146344), 'index': 14, 'word': 'Face', 'start': 55, 'end': 59}
{'entity': 'B-MISC', 'score': np.float32(0.67702544), 'index': 16, 'word': 'Google', 'start': 63, 'end': 69}
{'entity': 'I-MISC', 'score': np.float32(0.65608263), 'index': 17, 'word': 'Cola', 'start': 70, 'end': 74}
{'entity': 'I-MISC', 'score': np.float32(0.72049034), 'index': 18, 'word': '##b', 'start': 74, 'end': 75}
{'entity': 'B-LOC', 'score': np.float32(0.9998393), 'index': 20, 'word': 'São', 'start': 79, 'end': 82}
{'entity': 'I-LOC', 'score': np.float32(0.99960834), 'index': 21, 'word': 'Paulo', 'start': 83, 'end': 88}


In [9]:
# Resposta a Perguntas com Contexto (Question Answering)

question = "O que são os pipelines do Hugging Face?"
context = "Pipelines são uma API de alto nível para inferência de LLMs em tarefas comuns de processamento de linguagem natural."

question_answerer = pipeline("question-answering", model="pierreguillou/bert-base-cased-squad-v1.1-portuguese", device="cuda")
result = question_answerer(question=question, context=context)
print(result)

Device set to use cuda


{'score': 0.29969555139541626, 'start': 14, 'end': 35, 'answer': 'uma API de alto nível'}


In [10]:
# Sumarização de Texto

summarizer = pipeline("summarization", model="csebuetnlp/mT5_multilingual_XLSum", device="cuda")
text = """
A biblioteca transformers do Hugging Face é uma ferramenta incrivelmente versátil e poderosa para processamento de linguagem natural (PLN).
Ela permite aos usuários realizar uma ampla variedade de tarefas, como classificação de texto, reconhecimento de entidades nomeadas e resposta a perguntas.
É uma biblioteca extremamente popular, amplamente utilizada pela comunidade open-source de ciência de dados.
Ela reduz a barreira de entrada na área, fornecendo aos cientistas de dados uma forma produtiva de trabalhar com modelos transformers.
"""

summary = summarizer(text, max_length=50, min_length=25, do_sample=False)
print(summary[0]['summary_text'])

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Device set to use cuda


O Hugging Face é uma ferramenta que permite aos usuários realizar tarefas simples e complexas.


In [11]:
# Tradução (Português para Inglês)
translator = pipeline("translation", model="unicamp-dl/translation-pt-en-t5", device="cuda")
result = translator("Cientistas de dados ficaram impressionados com a simplicidade da API de pipelines do HuggingFace.")
print(result[0]['translation_text'])

Device set to use cuda
Your input_length: 26 is bigger than 0.9 * max_length: 20. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)


Data scientists were impressed by the simplicity of the HuggingFace pipelines IPA.


In [12]:
# Tradução (Inglês para Português)
translator = pipeline("translation", model="unicamp-dl/translation-en-pt-t5", device="cuda")
result = translator("The Data Scientists were truly amazed by the power and simplicity of the HuggingFace pipeline API.")
print(result[0]['translation_text'])

Device set to use cuda
Your input_length: 35 is bigger than 0.9 * max_length: 20. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)


Os cientistas dos dados foram realmente impressionados com o poder e a simplicidade da pipeline HuggingFace API.


In [13]:
# Classificação (Zero-Shot)

classifier = pipeline("zero-shot-classification", model="joeddav/xlm-roberta-large-xnli", device="cuda")
result = classifier("A biblioteca Transformers do Hugging Face é fantástica!", candidate_labels=["tecnologia", "esportes", "política"])
print(result)

Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda


{'sequence': 'A biblioteca Transformers do Hugging Face é fantástica!', 'labels': ['tecnologia', 'esportes', 'política'], 'scores': [0.6757574081420898, 0.2695833742618561, 0.05465933680534363]}


In [14]:
# Geração de Texto

generator = pipeline("text-generation", model="pierreguillou/gpt2-small-portuguese", device="cuda")
result = generator("Se há uma coisa importante sobre os pipelines do HuggingFace, é que")
print(result[0]['generated_text'])

Device set to use cuda


Se há uma coisa importante sobre os pipelines do HuggingFace, é que ela é uma espécie de "stripper" em um universo onde tudo funciona de forma igual, e não se move mais rápido do que as outras. Ela é uma forma de "Stripper", e ela não é mais uma forma de "stripper".

O HuggingFace foi descrito por Richard Dawkins em sua palestra sobre religião em Cambridge, na qual Dawkins disse que "os pipels do corpo humano são um tipo de "stripper" de um mundo onde tudo funciona de forma igual".

Os pipels de HuggingFace são muito similares aos pipelines de um cadáver e são encontrados nas florestas e nas árvores e em outras superfícies. Eles são tão semelhantes que, em alguns lugares, a palavra "pipeline" é combinada com pipeline e "stripper" com "stripper". O significado de "pipeline" é "para ser encontrado", e não "para ser encontrado". Outra palavra que se refere a pipelines de uma cabeça é "pipeline-de-cabeça", e, em alguns lugares, ela é combinada com o termo "pipeline" e "stripper".

O Huggin

In [15]:
# Geração de Áudio (Text-to-Speech)

from transformers import pipeline
from datasets import load_dataset
import soundfile as sf
import torch
from IPython.display import Audio

synthesiser = pipeline("text-to-speech", "microsoft/speecht5_tts", device='cuda')
embeddings_dataset = load_dataset("matthijs/cmu-arctic-xvectors", split="validation", trust_remote_code=True)
speaker_embedding = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)
speech = synthesiser("Olá! Este é um teste de síntese de voz para engenheiros de inteligência artificial.", forward_params={"speaker_embeddings": speaker_embedding})

Audio(speech["audio"], rate=speech["sampling_rate"])

Device set to use cuda


## Referências de Pipelines Disponíveis

Lista de referência para consultar os pipelines disponíveis nas bibliotecas Transformers e Diffusers:

- **Transformers Pipelines (Tarefas de PNL, Visão e Áudio):**
  [Documentação do Hugging Face Transformers](https://huggingface.co/docs/transformers/main_classes/pipelines)

- **Diffusers Pipelines (Modelos de Difusão):**
  [Documentação do Hugging Face Diffusers](https://huggingface.co/docs/diffusers/en/api/pipelines/overview)